# Tratamento Silver

Lê a tabela `bronze.vb_matches` e aplica os tratamentos definidos no diagnóstico de qualidade: `"NA"` convertido em NULL, tipagem das colunas, altura em cm, ranking separado em seed principal e de qualificatória, placar decomposto em sets, idade recalculada a partir do nascimento e flags de exceção, sem excluir nenhum linha

## 1. "NA" → NULL
Toda ausência na fonte é o texto `"NA"`, sem nenhum NULL real. A conversão vem antes de
qualquer cast.

Diagnóstico: 3.

In [0]:
from pyspark.sql import functions as F

# Bronze completa; separa as colunas de dados dos metadados técnicos (_ingestao_ts, _arquivo_origem, _camada)
bronze = spark.table("workspace.bronze.vb_matches")
colunas = [c for c in bronze.columns if not c.startswith("_")]

# Toda ausência na fonte é o texto "NA", sem NULL real (diagnóstico, seção 3).
df = bronze.select(*[F.when(F.col(c) == "NA", None).otherwise(F.col(c)).alias(c) for c in colunas])

## 2. Chave, tipagem e flags da partida

### 2.1. Chave da partida — diagnóstico 2.1

`id_partida` é o hash de `tournament`, `date`, `gender`, `bracket` e `match_num`, a chave
natural mínima de uma fonte que não traz identificador.

In [0]:
# Colunas da chave natural
chave_partida = ["tournament", "date", "gender", "bracket", "match_num"]

# id_partida: hash determinístico da chave natural. O bitwiseAND zera o bit de sinal,
# porque xxhash64 é assinado e devolveria metade dos ids negativos.
base = df.withColumn("id_partida", F.xxhash64(*chave_partida).bitwiseAND(F.lit(9223372036854775807)))

# OBS: hash não garante unicidade, mas em 64 bits com 76.756 linhas a colisão é da ordem de 1e-10
# (razoável para um dataset estático), e a seção 6 verifica a cada carga.
# O ideal seria um key map com ids sequenciais persistidos; ficou como trabalho futuro.

### 2.2. Tipagem e placar — diagnóstico 4.1 e 5.4

Converte data, ano, número da partida e duração para seus tipos, e decompõe o placar em
sets ganhos por cada dupla.

In [0]:
# 1. Placar regular: só pares "a-b" separados por vírgula ("21-19, 18-21, 15-13").
# "Forfeit or other" e "... retired" não batem e ficam como irregulares; as 22 sem placar (NULL) tmb
placar_regular = F.coalesce(F.col("score").rlike(r"^(\d+-\d+)(, \d+-\d+)*$"), F.lit(False))

# Sets ganhos: conta os pares "a-b" em que a > b (vencedor) ou a < b (perdedor)
def sets_ganhos(comparador):
    return F.expr(
        "size(filter(split(score, ', '), "
        f"s -> try_cast(split(s, '-')[0] as int) {comparador} try_cast(split(s, '-')[1] as int)))"
    )

# 2. Duração "hh:mm:ss" → minutos inteiros (horas × 60 + minutos); segundos são descartados.
duracao_min = F.expr(
    "cast(split(duration, ':')[0] as int) * 60 + cast(split(duration, ':')[1] as int)"
)

# Aplicando tipagem correta para o tipo de dado
base = (
    base.withColumn("data", F.to_date("date"))  # 100% no formato ISO, to_date direto
        .withColumn("ano", F.col("year").cast("int"))
        .withColumn("num_partida", F.col("match_num").cast("int"))
        .withColumn("duracao_min", duracao_min)
        .withColumn("placar_regular", placar_regular)
        .withColumn("sets_vencedor", F.when(placar_regular, sets_ganhos(">")))  # NULL quando o placar não é regular
        .withColumn("sets_perdedor", F.when(placar_regular, sets_ganhos("<")))
)

### 2.3. Ranking e fase do torneio — diagnóstico: 4.1 e 4.2.

Separa o ranking em seed principal e de qualificatória, e agrupa os 36 valores de
`bracket` em grupos, qualificatória e eliminatória.

In [0]:
# Seed principal, ou seja, "7" → 7 | "Q9" → NULL | "24, Q30" → 24
def seed_principal(col):
    return F.expr(
        f"cast(get(filter(split({col}, ', '), x -> x not rlike '^Q'), 0) as int)"
    )

# Seed da qualificatória, ou seja, "7" → NULL | "Q9" → 9 | "24, Q30" → 30
def seed_qualificatoria(col):
    return F.expr(
        f"cast(regexp_replace(get(filter(split({col}, ', '), x -> x rlike '^Q'), 0), 'Q', '') as int)"
    )

base = (
    base.withColumn("w_seed_principal", seed_principal("w_rank"))
        .withColumn("w_seed_qualificatoria", seed_qualificatoria("w_rank"))
        .withColumn("l_seed_principal", seed_principal("l_rank"))
        .withColumn("l_seed_qualificatoria", seed_qualificatoria("l_rank"))
)


# Pool* são grupos; Qualifier* e Country Quota são qualificatória; o restante é eliminatória
fase = (
    F.when(F.col("bracket").rlike("^Pool"), "grupos")
     .when(F.col("bracket").rlike("^Qualifier|^Country Quota"), "qualificatoria")
     .otherwise("eliminatoria")
)

base = base.withColumn("fase", fase)

# Mapeamento bracket → fase completo: o otherwise() não avisa, então os 36 valores precisam ser conferidos
display(base.groupBy("fase", "bracket").count().orderBy("fase", "bracket"))

### 2.4. Flags de exceção — diagnóstico 4.1, 5.3, 5.4 e 5.5

Marca partida sem resultado utilizável (incompleta, sem placar ou com a mesma dupla dos
dois lados), duração implausível e placar que contradiz o vencedor, sem excluir nenhuma
linha.

In [0]:
# Dupla dos dois lados: bye, W.O. ou erro de registro
mesma_dupla = F.size(F.array_intersect(
    F.array("w_player1", "w_player2"), F.array("l_player1", "l_player2"))) > 0

# W.O., desistência, placar ausente ou dupla repetida nos dois lados (vencerdor e perdedor)
flag_partida_incompleta = ~F.col("placar_regular") | mesma_dupla

# Abaixo de 15 min não dá para completar dois sets; coalesce porque 2.249 partidas não têm duração
flag_duracao_suspeita = F.coalesce(F.col("duracao_min") < 15, F.lit(False))

# Placar regular que contradiz o vencedor: 3 sets do vencedor, ou perdedor com mais sets
flag_placar_inconsistente = F.col("placar_regular") & ~(
    (F.col("sets_vencedor") > F.col("sets_perdedor")) & (F.col("sets_vencedor") <= 2)
)

base = (
    base.withColumn("flag_partida_incompleta", flag_partida_incompleta)
        .withColumn("flag_duracao_suspeita", flag_duracao_suspeita)
        .withColumn("flag_placar_inconsistente", flag_placar_inconsistente)
)

## 3. Carga da `silver.partida`

Reúne as colunas que descrevem a partida, renomeadas para português, e grava a tabela.


In [0]:
partida = base.select(
    "id_partida",
    # Onde e quando: torneio, sede, data e gênero da chave
    F.col("circuit").alias("circuito"),
    F.col("tournament").alias("torneio"),
    F.col("country").alias("pais_torneio"),
    "ano", "data",
    F.col("gender").alias("genero"),
    # Estrutura do torneio
    "num_partida",
    F.col("bracket").alias("chave"),
    F.col("round").alias("rodada"),
    "fase",
    # Resultado
    F.col("score").alias("placar"),
    "placar_regular", "sets_vencedor", "sets_perdedor",
    "duracao_min",
    # Seeds das duas duplas, na principal e na qualificatória
    "w_seed_principal", "w_seed_qualificatoria",
    "l_seed_principal", "l_seed_qualificatoria",
    # Flags, sem exclusão de linha
    "flag_partida_incompleta", "flag_duracao_suspeita", "flag_placar_inconsistente",
    F.current_timestamp().alias("_processado_em"),
)

(partida.write.format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.partida"))

p = spark.table("workspace.silver.partida")

display(p.limit(5))

## 4. Carga da `silver.atleta_partida` — diagnóstico 3, 5.1 e 5.2

Desempilha as quatro posições de cada partida em uma linha por atleta e lado, converte a
altura para cm, recalcula a idade a partir do nascimento e grava a tabela.


In [0]:
# Colunas de estátisticas de jogo
estatisticas = ["tot_attacks", "tot_kills", "tot_errors", "tot_hitpct",
                "tot_aces", "tot_serve_errors", "tot_blocks", "tot_digs"]

# Uma posição da fonte por vez, sempre com as mesmas colunas, para empilhar depois
def posicao(lado, n):
    p = f"{lado}_p{n}"
    return base.select(
        # Vínculo com a partida
        "id_partida", "data", F.col("gender").alias("genero"),
        # Lado do resultado
        F.lit(lado == "w").alias("vencedor"),
        # Atleta; nome sem espaço duplo nem espaço nas pontas
        F.regexp_replace(F.trim(F.col(f"{lado}_player{n}")), " +", " ").alias("nome"),
        F.to_date(F.col(f"{p}_birthdate")).alias("nascimento"),
        F.expr(f"try_cast({p}_hgt as int)").alias("altura_in"),
        F.col(f"{p}_country").alias("pais"),
        # Estatísticas de jogo; try_cast porque a 4.1 mediu só a posição w_p1
        *[F.expr(f"try_cast({p}_{e} as double)").alias(e) for e in estatisticas],
    )

atleta_partida = (
    posicao("w", 1).unionByName(posicao("w", 2))
    .unionByName(posicao("l", 1)).unionByName(posicao("l", 2))
    # id_atleta: mesmo critério do id_partida, sobre nome normalizado + nascimento
    .withColumn("id_atleta", F.xxhash64(F.lower("nome"), "nascimento").bitwiseAND(F.lit(9223372036854775807)))
    # Derivadas
    .withColumn("altura_cm", F.round(F.col("altura_in") * 2.54).cast("int"))
    .withColumn("idade_na_partida", F.floor(F.months_between("data", "nascimento") / 12).cast("int"))
    .withColumn("tem_estatistica", F.col("tot_attacks").isNotNull())
    # Exceções marcadas, sem exclusão; coalesce porque ausência não é exceção
    .withColumn("flag_estatistica_invalida", F.coalesce(
        (F.col("tot_attacks") < 0) | (F.col("tot_hitpct") > 1) | (F.col("tot_hitpct") < -1),
        F.lit(False)))
    .withColumn("flag_idade_atipica", F.coalesce(
        (F.col("idade_na_partida") < 15) | (F.col("idade_na_partida") > 50),
        F.lit(False)))
    .drop("altura_in")
    .withColumn("_processado_em", F.current_timestamp())
)

(atleta_partida.write.format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.atleta_partida"))

ap = spark.table("workspace.silver.atleta_partida")
display(ap.limit(5))

## 5. Validação pós-carga

Confere as chaves primárias, a integridade entre as duas tabelas e que nenhuma linha foi
perdida no caminho.

In [0]:
p  = spark.table("workspace.silver.partida")
ap = spark.table("workspace.silver.atleta_partida")

# Chaves primárias únicas
assert p.select("id_partida").distinct().count() == p.count(), "id_partida duplicado"
assert ap.select("id_partida", "id_atleta", "vencedor").distinct().count() == ap.count(), \
    "(id_partida, id_atleta, vencedor) duplicado"

# Integridade: toda participação aponta para uma partida existente
assert ap.join(p, "id_partida", "left_anti").count() == 0, "participação sem partida"

# Nada foi excluído: 4 atletas por partida
assert ap.count() == 4 * p.count(), "unpivot perdeu linhas"

print(f"partida {p.count():,} | atleta_partida {ap.count():,} — tudo consistente")

## Resumo dos tratamentos

| Tratamento | Onde |
|---|---|
| `"NA"` → NULL em todas as colunas | as três tabelas |
| Tipagem: datas, inteiros, duração em minutos | partida, desempenho |
| Ranking `Q16` / `"10, Q1"` → `seed_principal` + `seed_qualificatoria` | partida |
| Placar parseado em sets; `flag_partida_incompleta` para forfeit/retired | partida |
| `fase` derivada de `bracket` (grupos / qualificatória / eliminatória) | partida |
| Unpivot das 4 posições; altura polegada → cm; idade recalculada de nascimento × data | desempenho |
| Flags `estatistica_invalida` (32) e `idade_atipica` (134), sem excluir | desempenho |
| Dedup de atleta por nome normalizado + nascimento; 791 sem nascimento | atleta |

**Limitação conhecida:** atleta com nascimento ausente em parte das partidas vira dois `id_atleta` (4 casos identificados).